The following code was run on 13/11/24 to generate the Australian genomics data CSV file.

In [ ]:
import pandas as pd
from datetime import datetime

In [ ]:
def get_strain_data(nextclade, country):
    url = f"https://github.com/hodcroftlab/covariants/raw/refs/heads/master/cluster_tables/{nextclade}_data.json"
    json_data = pd.read_json(url)[country]
    data = pd.DataFrame(
        {
            "total": json_data["total_sequences"], 
            "variant": json_data["cluster_sequences"],
        }, index=json_data["week"]
    )
    data.index = pd.to_datetime(data.index)
    data["prop"] = data["variant"] / data["total"]
    return data

In [ ]:
def get_aust_var_data():
    var_map = {
        "ba1": "21K.Omicron",
        "ba2": "21L.Omicron",
        "ba5": "22B.Omicron",
    }
    var_data = {k: get_strain_data(v, "Australia")["prop"] for k, v in var_map.items()}
    return pd.DataFrame(var_data)

In [ ]:
var_props = get_aust_var_data()

In [ ]:
var_props.plot.area()

In [ ]:
var_props.to_csv("../data/covid_aus/var.csv")

In [ ]:
norm_props = var_props.div(var_props.sum(axis=1), axis=0).dropna()

In [ ]:
norm_props.loc[norm_props.index < datetime(2022, 10, 1)].plot.area()